In [1]:
import pandas as pd
import numpy as np

In [2]:
DATA_PATH = "C:\\Users\\HP\\OneDrive\\المستندات\\GitHub\\Student-Leak-Radar\\project\\data\\raw\\"

student_info = pd.read_csv(DATA_PATH + "studentInfo.csv")
courses = pd.read_csv(DATA_PATH + "courses.csv")
student_vle = pd.read_csv(DATA_PATH + "studentVle.csv")
vle = pd.read_csv(DATA_PATH + "vle.csv")
assessments = pd.read_csv(DATA_PATH + "assessments.csv")
student_assessment = pd.read_csv(DATA_PATH + "studentAssessment.csv")

In [3]:
base = student_info.copy()

base = base[base["studied_credits"] <= 250].copy()

base["target"] = base["final_result"].map({
    "Distinction": "successful",
    "Pass": "successful",
    "Fail": "at_risk",
    "Withdrawn": "at_risk"
})

base = base.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

base.shape

(32480, 14)

In [4]:
student_vle_full = student_vle.merge(
    vle,
    on=["id_site", "code_module", "code_presentation"],
    how="left"
)

student_vle_full = student_vle_full.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

student_vle_full["relative_time"] = (
    student_vle_full["date"] / student_vle_full["module_presentation_length"]
)

student_vle_full.head()

,code_module,code_presentation,id_student,id_site,date,sum_click,activity_type,week_from,week_to,module_presentation_length,relative_time
0,AAA,2013J,28400,546652,-10,4,forumng,NaN,NaN,268,-0.037313
1,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN,268,-0.037313
2,AAA,2013J,28400,546652,-10,1,forumng,NaN,NaN,268,-0.037313
3,AAA,2013J,28400,546614,-10,11,homepage,NaN,NaN,268,-0.037313
4,AAA,2013J,28400,546714,-10,1,oucontent,NaN,NaN,268,-0.037313


In [5]:
vle_features = student_vle_full.groupby(
    ["id_student", "code_module", "code_presentation"]
).agg(
    total_clicks=("sum_click", "sum"),
    total_active_days=("date", "nunique"),
    unique_sites=("id_site", "nunique"),
    unique_activity_types=("activity_type", "nunique"),
    clicks_first_25=("sum_click", lambda x: x[student_vle_full.loc[x.index, "relative_time"] <= 0.25].sum()),
    clicks_first_50=("sum_click", lambda x: x[student_vle_full.loc[x.index, "relative_time"] <= 0.50].sum()),
    clicks_last_25=("sum_click", lambda x: x[student_vle_full.loc[x.index, "relative_time"] >= 0.75].sum())
).reset_index()

vle_features["clicks_per_active_day"] = (
    vle_features["total_clicks"] / vle_features["total_active_days"]
).replace([np.inf, -np.inf], 0).fillna(0)

vle_features.head()

,id_student,code_module,code_presentation,total_clicks,total_active_days,unique_sites,unique_activity_types,clicks_first_25,clicks_first_50,clicks_last_25,clicks_per_active_day
0,6516,AAA,2014J,2791,159,84,7,1118,1603,469,17.553459
1,8462,DDD,2013J,646,56,125,9,527,646,0,11.535714
2,8462,DDD,2014J,10,1,3,3,10,10,0,10.000000
3,11391,AAA,2013J,934,40,55,6,545,710,186,23.350000
4,23629,BBB,2013B,161,16,11,5,119,161,0,10.062500


In [6]:
activity_features = student_vle_full.pivot_table(
    index=["id_student", "code_module", "code_presentation"],
    columns="activity_type",
    values="sum_click",
    aggfunc="sum",
    fill_value=0
).reset_index()

activity_features.columns.name = None

activity_features.head()

,id_student,code_module,code_presentation,dataplus,dualpane,externalquiz,folder,forumng,glossary,homepage,...,ouelluminate,ouwiki,page,questionnaire,quiz,repeatactivity,resource,sharedsubpage,subpage,url
0,6516,AAA,2014J,21,0,0,0,451,0,497,...,0,0,0,0,0,0,31,0,143,143
1,8462,DDD,2013J,0,0,12,0,36,0,184,...,0,18,0,0,0,0,70,0,227,23
2,8462,DDD,2014J,0,0,0,0,2,0,7,...,0,0,0,0,0,0,0,0,0,0
3,11391,AAA,2013J,0,0,0,0,193,0,138,...,0,0,0,0,0,0,13,0,32,5
4,23629,BBB,2013B,0,0,0,0,87,0,36,...,0,0,0,0,31,0,2,0,5,0


In [7]:
assessment_full = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)

assessment_full = assessment_full.merge(
    courses,
    on=["code_module", "code_presentation"],
    how="left"
)

assessment_full["assessment_relative_time"] = (
    assessment_full["date"] / assessment_full["module_presentation_length"]
)

assessment_full["submitted_relative_time"] = (
    assessment_full["date_submitted"] / assessment_full["module_presentation_length"]
)

assessment_full["is_late"] = assessment_full["date_submitted"] > assessment_full["date"]

assessment_full["submission_delay"] = (
    assessment_full["date_submitted"] - assessment_full["date"]
)

assessment_full.head()

,id_assessment,id_student,date_submitted,is_banked,score,code_module,code_presentation,assessment_type,date,weight,module_presentation_length,assessment_relative_time,submitted_relative_time,is_late,submission_delay
0,1752,11391,18,0,78.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.067164,False,-1.0
1,1752,28400,22,0,70.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.082090,True,3.0
2,1752,31604,17,0,72.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.063433,False,-2.0
3,1752,32885,26,0,69.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.097015,True,7.0
4,1752,38053,19,0,79.0,AAA,2013J,TMA,19.0,10.0,268,0.070896,0.070896,False,0.0


In [8]:
assessment_features = assessment_full.groupby(
    ["id_student", "code_module", "code_presentation"]
).agg(
    avg_score=("score", "mean"),
    min_score=("score", "min"),
    max_score=("score", "max"),
    submitted_assessments=("id_assessment", "count"),
    late_submissions=("is_late", "sum"),
    avg_submission_delay=("submission_delay", "mean"),
    assessments_first_50=("assessment_relative_time", lambda x: (x <= 0.50).sum())
).reset_index()

assessment_features.head()

,id_student,code_module,code_presentation,avg_score,min_score,max_score,submitted_assessments,late_submissions,avg_submission_delay,assessments_first_50
0,6516,AAA,2014J,61.800000,48.0,77.0,5,0,-2.600000,3
1,8462,DDD,2013J,87.666667,83.0,93.0,3,1,-0.333333,3
2,8462,DDD,2014J,86.500000,83.0,93.0,4,0,-59.500000,4
3,11391,AAA,2013J,82.000000,78.0,85.0,5,0,-1.800000,3
4,23629,BBB,2013B,82.500000,63.0,100.0,4,3,3.500000,4


In [9]:
final_df = base.merge(
    vle_features,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

final_df = final_df.merge(
    activity_features,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

final_df = final_df.merge(
    assessment_features,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

final_df.shape

(32480, 49)

In [10]:
numeric_cols = final_df.select_dtypes(include=["number"]).columns

final_df[numeric_cols] = final_df[numeric_cols].fillna(0)

final_df["imd_band"] = final_df["imd_band"].fillna("Unknown")

final_df.isnull().sum().sort_values(ascending=False).head(20)

code_module                   0
code_presentation             0
id_student                    0
gender                        0
region                        0
highest_education             0
imd_band                      0
age_band                      0
num_of_prev_attempts          0
studied_credits               0
disability                    0
final_result                  0
target                        0
module_presentation_length    0
total_clicks                  0
total_active_days             0
unique_sites                  0
unique_activity_types         0
clicks_first_25               0
clicks_first_50               0
dtype: int64

In [11]:
FULL_LOAD_OU = 120

final_df["workload_ratio"] = final_df["studied_credits"] / FULL_LOAD_OU
final_df["workload_ratio"] = final_df["workload_ratio"].clip(upper=2.0)

final_df["workload_level"] = pd.cut(
    final_df["workload_ratio"],
    bins=[0, 0.5, 1.0, 1.5, 2.0],
    labels=["light", "normal", "high", "very_high"],
    include_lowest=True
)

final_df[["studied_credits", "workload_ratio", "workload_level"]].head()

,studied_credits,workload_ratio,workload_level
0,240,2.0,very_high
1,60,0.5,light
2,60,0.5,light
3,60,0.5,light
4,60,0.5,light


In [12]:
from pathlib import Path

output_path = Path("data/processed/student_course_features.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

final_df.to_csv(output_path, index=False)

print("Final dataset saved successfully.")
print("Shape:", final_df.shape)

Final dataset saved successfully.
Shape: (32480, 51)


In [13]:
student_vle_full
assessment_full
base

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,target,module_presentation_length
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,successful,268
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,successful,268
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,at_risk,268
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,successful,268
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,successful,268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32475,GGG,2014J,2640965,F,Wales,Lower Than A Level,10-20,0-35,0,30,N,Fail,at_risk,269
32476,GGG,2014J,2645731,F,East Anglian Region,Lower Than A Level,40-50%,35-55,0,30,N,Distinction,successful,269
32477,GGG,2014J,2648187,F,South Region,A Level or Equivalent,20-30%,0-35,0,30,Y,Pass,successful,269
32478,GGG,2014J,2679821,F,South East Region,Lower Than A Level,90-100%,35-55,0,30,N,Withdrawn,at_risk,269


In [14]:
def build_early_warning_dataset(cutoff, output_name):
    
    # 1) VLE data حتى نقطة معينة من الفصل
    vle_until_cutoff = student_vle_full[
        student_vle_full["relative_time"] <= cutoff
    ].copy()

    vle_features = vle_until_cutoff.groupby(
        ["id_student", "code_module", "code_presentation"]
    ).agg(
        total_clicks_until_cutoff=("sum_click", "sum"),
        active_days_until_cutoff=("date", "nunique"),
        unique_sites_until_cutoff=("id_site", "nunique"),
        unique_activity_types_until_cutoff=("activity_type", "nunique")
    ).reset_index()

    vle_features["clicks_per_active_day_until_cutoff"] = (
        vle_features["total_clicks_until_cutoff"] /
        vle_features["active_days_until_cutoff"]
    ).replace([np.inf, -np.inf], 0).fillna(0)

    # 2) Activity type features حتى نقطة معينة
    activity_features = vle_until_cutoff.pivot_table(
        index=["id_student", "code_module", "code_presentation"],
        columns="activity_type",
        values="sum_click",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    activity_features.columns.name = None

    # 3) Assessment data حتى نقطة معينة
    assessment_until_cutoff = assessment_full[
        assessment_full["submitted_relative_time"] <= cutoff
    ].copy()

    assessment_features = assessment_until_cutoff.groupby(
        ["id_student", "code_module", "code_presentation"]
    ).agg(
        avg_score_until_cutoff=("score", "mean"),
        min_score_until_cutoff=("score", "min"),
        max_score_until_cutoff=("score", "max"),
        submitted_assessments_until_cutoff=("id_assessment", "count"),
        late_submissions_until_cutoff=("is_late", "sum"),
        avg_submission_delay_until_cutoff=("submission_delay", "mean")
    ).reset_index()

    # 4) الدمج النهائي
    final_early_df = base.merge(
        vle_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    final_early_df = final_early_df.merge(
        activity_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    final_early_df = final_early_df.merge(
        assessment_features,
        on=["id_student", "code_module", "code_presentation"],
        how="left"
    )

    # 5) تعبئة القيم الفارغة الرقمية
    numeric_cols = final_early_df.select_dtypes(include=["number"]).columns
    final_early_df[numeric_cols] = final_early_df[numeric_cols].fillna(0)

    # 6) معالجة imd_band
    final_early_df["imd_band"] = final_early_df["imd_band"].fillna("Unknown")

    # 7) Workload features
    FULL_LOAD_OU = 120

    final_early_df["workload_ratio"] = (
        final_early_df["studied_credits"] / FULL_LOAD_OU
    ).clip(upper=2.0)

    final_early_df["workload_level"] = pd.cut(
        final_early_df["workload_ratio"],
        bins=[0, 0.5, 1.0, 1.5, 2.0],
        labels=["light", "normal", "high", "very_high"],
        include_lowest=True
    )

    # 8) حفظ الملف
    from pathlib import Path

    output_path = Path(f"data/processed/{output_name}")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    final_early_df.to_csv(output_path, index=False)

    print(f"Saved: {output_path}")
    print("Shape:", final_early_df.shape)

    return final_early_df

In [15]:
df_25 = build_early_warning_dataset(
    cutoff=0.25,
    output_name="student_features_first_25.csv"
)

Saved: data\processed\student_features_first_25.csv
Shape: (32480, 46)


In [16]:
df_50 = build_early_warning_dataset(
    cutoff=0.50,
    output_name="student_features_first_50.csv"
)

Saved: data\processed\student_features_first_50.csv
Shape: (32480, 46)


In [17]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

numeric_df = df_50.select_dtypes(include=["number"]).copy()

if "id_student" in numeric_df.columns:
    numeric_df = numeric_df.drop(columns=["id_student"])

corr_matrix = numeric_df.corr()

fig = go.Figure(
    data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.columns,
        colorscale="RdBu",
        zmin=-1,
        zmax=1,
        colorbar=dict(title="Correlation")
    )
)

fig.update_layout(
    title="Correlation Heatmap - Numeric Features",
    width=1000,
    height=900
)

fig.show()

In [18]:
threshold = 0.80

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_pairs = upper_triangle.stack().reset_index()
high_corr_pairs.columns = ["feature_1", "feature_2", "correlation"]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["correlation"].abs() >= threshold
].sort_values(by="correlation", ascending=False)

display(high_corr_pairs)

,feature_1,feature_2,correlation
64,studied_credits,workload_ratio,0.999997
541,avg_score_until_cutoff,max_score_until_cutoff,0.975989
540,avg_score_until_cutoff,min_score_until_cutoff,0.950346
105,total_clicks_until_cutoff,homepage,0.880025
546,min_score_until_cutoff,max_score_until_cutoff,0.872046
174,unique_sites_until_cutoff,subpage,0.850421
96,total_clicks_until_cutoff,active_days_until_cutoff,0.818072
